# Análisis de accidentes cerebrovasculares (ACV)

Análisis de 5.110 pacientes mediante consultas SQL, minería predictiva con Random Forest y minería descriptiva con K-Means.

In [0]:

%sql
SELECT *
FROM workspace.default.healthcare_dataset_stroke_data


id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,N/A,never smoked,1
31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24,never smoked,1
56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29,formerly smoked,1
53882,Male,74.0,1,1,Yes,Private,Rural,70.09,27.4,never smoked,1
10434,Female,69.0,0,0,No,Private,Urban,94.39,22.8,never smoked,1
27419,Female,59.0,0,0,Yes,Private,Rural,76.15,N/A,Unknown,1
60491,Female,78.0,0,0,Yes,Private,Urban,58.57,24.2,Unknown,1


In [0]:

%sql
SELECT
    CASE
        WHEN stroke = 1 THEN 'Con ACV'
        ELSE 'Sin ACV'
    END AS Estado_ACV,
    COUNT(*) AS Total_Pacientes
FROM workspace.default.healthcare_dataset_stroke_data
GROUP BY stroke
ORDER BY stroke DESC;


Estado_ACV,Total_Pacientes
Con ACV,249
Sin ACV,4861


Databricks visualization. Run in Databricks to view.

In [0]:

%sql
SELECT
    CASE
        WHEN age < 30 THEN 'Menores de 30'
        WHEN age < 45 THEN '30-44 años'
        WHEN age < 60 THEN '45-59 años'
        WHEN age < 75 THEN '60-74 años'
        ELSE '75 años o más'
    END AS Grupo_Edad,
    COUNT(*) AS Total_Pacientes,
    SUM(stroke) AS Pacientes_Con_ACV,
    ROUND(100.0 * SUM(stroke) / COUNT(*), 2)
        AS Porcentaje_ACV
FROM workspace.default.healthcare_dataset_stroke_data
GROUP BY
    CASE
        WHEN age < 30 THEN 'Menores de 30'
        WHEN age < 45 THEN '30-44 años'
        WHEN age < 60 THEN '45-59 años'
        WHEN age < 75 THEN '60-74 años'
        ELSE '75 años o más'
    END
ORDER BY MIN(age);


Grupo_Edad,Total_Pacientes,Pacientes_Con_ACV,Porcentaje_ACV
Menores de 30,1515,2,0.13
30-44 años,1018,8,0.79
45-59 años,1201,58,4.83
60-74 años,858,79,9.21
75 años o más,518,102,19.69


Databricks visualization. Run in Databricks to view.

In [0]:

%sql
SELECT
    CASE
        WHEN hypertension = 1 THEN 'Con hipertensión'
        ELSE 'Sin hipertensión'
    END AS Hipertension,
    COUNT(*) AS Total_Pacientes,
    SUM(stroke) AS Pacientes_Con_ACV,
    ROUND(100.0 * SUM(stroke) / COUNT(*), 2)
        AS Porcentaje_ACV
FROM workspace.default.healthcare_dataset_stroke_data
GROUP BY hypertension
ORDER BY hypertension DESC;


Hipertension,Total_Pacientes,Pacientes_Con_ACV,Porcentaje_ACV
Con hipertensión,498,66,13.25
Sin hipertensión,4612,183,3.97


Databricks visualization. Run in Databricks to view.

## Minería predictiva — Random Forest

In [0]:

import pandas as pd
import numpy as np

df = spark.table(
    "workspace.default.healthcare_dataset_stroke_data"
).toPandas()

display(df.head())

# Mostrar dimensiones
print("Filas y columnas:", df.shape)



id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,N/A,never smoked,1
31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24,never smoked,1


Filas y columnas: (5110, 12)


In [0]:

print("INFORMACIÓN DEL DATASET")
df.info()

print("\nVALORES FALTANTES")
print(df.isnull().sum())

print("\nVALORES DE BMI")
print(df["bmi"].value_counts().head(10))

print("\nDISTRIBUCIÓN DE ACV")
print(df["stroke"].value_counts())


INFORMACIÓN DEL DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                5110 non-null   object 
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 479.2+ KB

VALORES FALTANTES
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0

In [0]:

df_limpio = df.copy()

# Convertir BMI a tipo numérico
df_limpio["bmi"] = pd.to_numeric(
    df_limpio["bmi"],
    errors="coerce"
)

print("Valores faltantes después de la conversión:")
print(df_limpio.isnull().sum())


print("\nPromedio BMI:", round(df_limpio["bmi"].mean(), 2))
print("Mediana BMI:", df_limpio["bmi"].median())


print("\nDimensiones:", df_limpio.shape)


Valores faltantes después de la conversión:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Promedio BMI: 28.89
Mediana BMI: 28.1

Dimensiones: (5110, 12)


In [0]:
# Separar variables predictoras y variable objetivo
X = df_limpio.drop(columns=["id", "stroke"]).copy()
y = df_limpio["stroke"].copy()

# Reemplazar BMI faltante con la mediana (como preparación previa del ejemplo)
X["bmi"] = X["bmi"].fillna(X["bmi"].median())

print("Variables originales:", X.shape)
print("Categorías:", X.select_dtypes(include=["object", "category"]).columns.tolist())


Variables originales: (5110, 10)
Categorías: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


In [0]:
%pip install imbalanced-learn


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from imblearn.over_sampling import SMOTENC

# SMOTENC antes de las dummies, como en el notebook de clase
columnas_categoricas = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

smote = SMOTENC(
    k_neighbors=2,
    sampling_strategy=0.2,
    categorical_features=columnas_categoricas,
    random_state=42
)

X_smote, y_smote = smote.fit_resample(X, y)

print("Distribución antes del balanceo:")
print(y.value_counts())
print("\nDistribución después de SMOTENC:")
print(y_smote.value_counts())


Distribución antes del balanceo:
stroke
0    4861
1     249
Name: count, dtype: int64

Distribución después de SMOTENC:
stroke
0    4861
1     972
Name: count, dtype: int64


In [0]:
from sklearn.model_selection import train_test_split

# Crear dummies después de SMOTENC
X_dummies = pd.get_dummies(X_smote, dtype=int)

# División 70/30, como en el notebook de clasificación de clase
X_train_bal, X_test_final, y_train_bal, y_test = train_test_split(
    X_dummies,
    y_smote,
    test_size=0.3,
    stratify=y_smote,
    random_state=42
)

print("Entrenamiento:", X_train_bal.shape)
print("Prueba:", X_test_final.shape)
print("\nClases en entrenamiento:")
print(y_train_bal.value_counts())
print("\nClases en prueba:")
print(y_test.value_counts())


Entrenamiento: (4083, 21)
Prueba: (1750, 21)

Clases en entrenamiento:
stroke
0    3403
1     680
Name: count, dtype: int64

Clases en prueba:
stroke
0    1458
1     292
Name: count, dtype: int64


In [0]:
# Random Forest: mismos parámetros del notebook Mineria_Predictiva
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=150,
    max_samples=0.9,
    criterion='gini',
    max_depth=20,
    min_samples_leaf=10,
    random_state=42
)

# Entrenamiento con el 70% de los datos preparados
model_rf.fit(X_train_bal, y_train_bal)
print('Modelo Random Forest entrenado correctamente')

Modelo Random Forest entrenado correctamente


In [0]:
# Importancia de las características (como en el notebook de clase)
importances = model_rf.feature_importances_
feature_names = X_train_bal.columns
forest_importances = pd.Series(importances, index=feature_names)

# Matplotlib presentó errores en este Databricks: visualizar mediante display
importancias_rf = forest_importances.sort_values(ascending=False).reset_index()
importancias_rf.columns = ['Caracteristica', 'Importancia']
display(importancias_rf)

Caracteristica,Importancia
age,0.4514563246464218
avg_glucose_level,0.21205270307452884
bmi,0.11990087235650529
ever_married_Yes,0.03442807144724375
ever_married_No,0.026618909708908345
smoking_status_never smoked,0.017823015881022492
work_type_Private,0.013999845678828279
work_type_Self-employed,0.01269029904918926
gender_Male,0.012382136411589088
gender_Female,0.01230574096636718


In [0]:
# Evaluación de Random Forest con el 30% de los datos
from sklearn import metrics

Y_pred = model_rf.predict(X_test_final)
y_pred = Y_pred  # Mantiene la variable utilizada por la tabla de métricas del dashboard
class_names = ['Sin ACV', 'Con ACV']

# Matriz de confusión
cm = metrics.confusion_matrix(y_true=y_test, y_pred=Y_pred)
matriz = cm
matriz_tabla = pd.DataFrame(
    cm,
    index=['Real: sin ACV', 'Real: con ACV'],
    columns=['Predicho: sin ACV', 'Predicho: con ACV']
)
print('Matriz de confusión:')
display(matriz_tabla)

# Precision, recall, F1-score y exactitud, como en el cuaderno de clase
print('Reporte de clasificación:')
print(metrics.classification_report(
    y_true=y_test, y_pred=Y_pred,
    target_names=class_names, zero_division=0
))
accuracy = metrics.accuracy_score(y_test, Y_pred)
print('Exactitud:', round(accuracy * 100, 2), '%')

Matriz de confusión:


Predicho: sin ACV,Predicho: con ACV
1427,31
203,89


Reporte de clasificación:
              precision    recall  f1-score   support

     Sin ACV       0.88      0.98      0.92      1458
     Con ACV       0.74      0.30      0.43       292

    accuracy                           0.87      1750
   macro avg       0.81      0.64      0.68      1750
weighted avg       0.85      0.87      0.84      1750

Exactitud: 86.63 %


In [0]:
# Curva ROC del Random Forest (equivalente a RocCurveDisplay del notebook de clase)
# En Databricks la graficamos mediante el resultado display, sin Matplotlib
probabilidades_acv = model_rf.predict_proba(X_test_final)[:, 1]
fpr, tpr, umbrales = metrics.roc_curve(y_test, probabilidades_acv)
auc_roc = metrics.roc_auc_score(y_test, probabilidades_acv)

curva_roc = pd.DataFrame({
    'FPR': fpr,
    'TPR': tpr
})
print('Área bajo la curva ROC (AUC):', round(auc_roc, 4))
display(curva_roc)

Área bajo la curva ROC (AUC): 0.9043


FPR,TPR
0.0,0.0
0.0,0.003424657534246575
0.0,0.05136986301369863
0.0013717421124828531,0.05136986301369863
0.0013717421124828531,0.0684931506849315
0.00205761316872428,0.0684931506849315
0.00205761316872428,0.0821917808219178
0.003429355281207133,0.0821917808219178
0.003429355281207133,0.09246575342465753
0.00411522633744856,0.09246575342465753


## Minería descriptiva — K-Means


In [0]:

from sklearn.preprocessing import MinMaxScaler

# Crear una copia del dataset
data = df_limpio.copy()

# Eliminar ID y variable objetivo
data = data.drop(columns=["id", "stroke"])

# Reemplazar valores faltantes de BMI
data["bmi"] = data["bmi"].fillna(
    data["bmi"].median()
)

# Convertir variables categoricas a dummies
data = pd.get_dummies(data, dtype=int)

# Normalizar todas las variables
scaler = MinMaxScaler()
data_normalizada = scaler.fit_transform(data)

# Convertir nuevamente a DataFrame
data_normalizada = pd.DataFrame(
    data_normalizada,
    columns=data.columns,
    index=data.index
)

# Verificar resultados
print("Dimensiones:", data_normalizada.shape)
print("Valores faltantes:", data_normalizada.isnull().sum().sum())

display(data_normalizada.head())


Dimensiones: (5110, 21)
Valores faltantes: 0


age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0.81689453125,0.0,1.0,0.8012648878219925,0.3012600229095075,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
0.74365234375,0.0,0.0,0.6790231742221402,0.20389461626575028,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
0.9755859375,0.0,1.0,0.2345120487489613,0.2542955326460481,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
0.59716796875,0.0,0.0,0.5360077555165728,0.27605956471935855,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
0.96337890625,1.0,0.0,0.5493490905733542,0.15693012600229095,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [0]:

from sklearn.cluster import KMeans

ks = range(2, 20)
inertias = []

for k in ks:
    model = KMeans(
        n_clusters=k,
        max_iter=300,
        random_state=42,
        n_init=10
    )

    model.fit(data_normalizada)
    inertias.append(model.inertia_)

# Crear tabla para graficar en Databricks
resultados_codo = pd.DataFrame({
    "Numero_Clusters": list(ks),
    "Inercia": inertias
})

display(resultados_codo)


Numero_Clusters,Inercia
2,12583.535217665667
3,10905.841277602192
4,9908.456385876052
5,9071.997418679532
6,8505.994878615304
7,7911.483740712735
8,7393.156983148421
9,7199.832102012616
10,6936.597323414341
11,6545.189404965384


Databricks visualization. Run in Databricks to view.

In [0]:

from sklearn.cluster import KMeans

# Crear el modelo K-Means
modelo_kmeans = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

# Entrenar el modelo
modelo_kmeans.fit(data_normalizada)

# Asignar los clusters
data["Cluster"] = modelo_kmeans.labels_



In [0]:

# Obtener los centroides
modelo_kmeans.cluster_centers_

# Crear DataFrame con los centroides
centroides = pd.DataFrame(
    modelo_kmeans.cluster_centers_,
    columns=data_normalizada.columns.values
)

# Mostrar centroides redondeados
display(centroides.round(1))


age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0.6,0.1,0.1,0.2,0.2,1.0,-0.0,-0.0,0.0,1.0,0.0,0.0,1.0,0.0,-0.0,1.0,0.0,0.2,0.2,0.4,0.2
0.1,0.0,0.0,0.2,0.1,0.4,0.6,-0.0,1.0,-0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.5,0.9,0.0,0.0,0.0
0.7,0.1,0.0,0.2,0.2,1.0,-0.0,0.0,0.0,1.0,0.2,0.0,0.6,0.2,-0.0,-0.0,1.0,0.2,0.2,0.4,0.2
0.7,0.2,0.1,0.3,0.2,-0.0,1.0,-0.0,0.0,1.0,0.2,0.0,0.7,0.2,-0.0,1.0,0.0,0.2,0.3,0.4,0.2
0.4,0.0,0.0,0.2,0.2,0.6,0.4,0.0,1.0,-0.0,0.1,0.0,0.8,0.1,0.0,1.0,0.0,0.2,0.1,0.5,0.2
0.7,0.1,0.1,0.3,0.2,-0.0,1.0,-0.0,0.0,1.0,0.2,0.0,0.6,0.2,-0.0,-0.0,1.0,0.2,0.3,0.3,0.2
0.4,0.1,0.0,0.2,0.2,0.7,0.3,-0.0,1.0,-0.0,0.1,0.0,0.7,0.1,0.0,-0.0,1.0,0.2,0.1,0.5,0.2
0.7,0.1,0.0,0.2,0.2,1.0,-0.0,-0.0,0.0,1.0,0.4,-0.0,0.0,0.6,0.0,1.0,0.0,0.2,0.2,0.5,0.1


In [0]:

# datos originales
perfilamiento = df_limpio.copy()

# Agregar el cluster asignado por K-Means
perfilamiento["Cluster"] = modelo_kmeans.labels_

# Ver los pacientes con sus clusters
display(perfilamiento.head(20))


print("Pacientes por cluster:")
print(perfilamiento["Cluster"].value_counts().sort_index())



id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,Cluster
9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,5
51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,null,never smoked,1,7
31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,3
60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,2
1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,7
56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1,5
53882,Male,74.0,1,1,Yes,Private,Rural,70.09,27.4,never smoked,1,3
10434,Female,69.0,0,0,No,Private,Urban,94.39,22.8,never smoked,1,6
27419,Female,59.0,0,0,Yes,Private,Rural,76.15,null,Unknown,1,0
60491,Female,78.0,0,0,Yes,Private,Urban,58.57,24.2,Unknown,1,2


Pacientes por cluster:
Cluster
0     609
1     679
2    1028
3     669
4     534
5     683
6     544
7     364
Name: count, dtype: int64


In [0]:

perfilamiento = df_limpio.copy()
perfilamiento["Cluster"] = modelo_kmeans.labels_

# Calcular promedios por cluster
perfil_final = perfilamiento.groupby("Cluster").agg({
    "age": "mean",
    "avg_glucose_level": "mean",
    "bmi": "mean",
    "hypertension": "mean",
    "heart_disease": "mean",
    "stroke": "mean"
}).round(2)

perfil_final["age"] = perfil_final["age"].round(0)

# Contar pacientes por cluster
perfil_final["Pacientes"] = perfilamiento.groupby(
    "Cluster"
).size()


display(perfil_final)


age,avg_glucose_level,bmi,hypertension,heart_disease,stroke,Pacientes
51.0,107.01,31.23,0.11,0.05,0.05,609
7.0,94.21,20.11,0.0,0.0,0.0,679
54.0,108.54,30.72,0.12,0.05,0.06,1028
55.0,115.57,30.93,0.17,0.11,0.07,669
31.0,99.52,28.63,0.04,0.02,0.01,534
56.0,116.19,31.09,0.13,0.11,0.08,683
32.0,96.23,28.3,0.06,0.03,0.04,544
58.0,108.61,29.99,0.15,0.04,0.08,364


## Interpretación de los 8 clusters

**Cluster 0 – Adultos de mediana edad:** Este grupo está formado por 609 pacientes, con una edad promedio de 51 años, un BMI de 31.23 y una proporción de ACV del 5%.

**Cluster 1 – Población infantil:** Contiene 679 pacientes, con una edad promedio de 7 años y un BMI de 20.11. Presenta la menor edad promedio y una proporción de ACV cercana al 0%.

**Cluster 2 – Adultos de mediana edad:** Es el grupo más numeroso, con 1.028 pacientes. Tiene una edad promedio de 54 años, una glucosa promedio de 108.54 y una proporción de ACV del 6%.

**Cluster 3 – Adultos con glucosa relativamente elevada:** Está compuesto por 669 pacientes, con una edad promedio de 55 años, glucosa de 115.57 y una proporción de ACV del 7%.

**Cluster 4 – Adultos jóvenes:** Agrupa 534 pacientes, con una edad promedio de 31 años, glucosa de 99.52 y una proporción de ACV del 1%.

**Cluster 5 – Adultos con mayor glucosa promedio:** Contiene 683 pacientes, con una edad promedio de 56 años y glucosa de 116.19. Presenta una proporción de ACV cercana al 8%.

**Cluster 6 – Adultos jóvenes:** Está formado por 544 pacientes, con una edad promedio de 32 años, glucosa de 96.23 y una proporción de ACV del 4%.

**Cluster 7 – Adultos de mayor edad promedio:** Incluye 364 pacientes, con una edad promedio de 58 años, un BMI de 29.99 y una proporción de ACV del 8%.




In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calcular las metricas del modelo actualmente entrenado
metricas_rf = pd.DataFrame({
    "Metrica": [
        "Accuracy (%)",
        "Precision ACV (%)",
        "Recall ACV (%)",
        "F1-score ACV (%)"
    ],
    "Valor": [
        accuracy_score(y_test, y_pred) * 100,
        precision_score(y_test, y_pred, zero_division=0) * 100,
        recall_score(y_test, y_pred, zero_division=0) * 100,
        f1_score(y_test, y_pred, zero_division=0) * 100
    ]
})
metricas_rf["Valor"] = metricas_rf["Valor"].round(2)
display(metricas_rf)


Metrica,Valor
Accuracy (%),86.63
Precision ACV (%),74.17
Recall ACV (%),30.48
F1-score ACV (%),43.2


In [0]:

display(perfil_final)


age,avg_glucose_level,bmi,hypertension,heart_disease,stroke,Pacientes
51.0,107.01,31.23,0.11,0.05,0.05,609
7.0,94.21,20.11,0.0,0.0,0.0,679
54.0,108.54,30.72,0.12,0.05,0.06,1028
55.0,115.57,30.93,0.17,0.11,0.07,669
31.0,99.52,28.63,0.04,0.02,0.01,534
56.0,116.19,31.09,0.13,0.11,0.08,683
32.0,96.23,28.3,0.06,0.03,0.04,544
58.0,108.61,29.99,0.15,0.04,0.08,364
